B17. Температура как каузальный рычаг разнообразия.....


In [1]:
import sys, os, re
from itertools import combinations

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/protocol_design')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
TMP = sweep_arms('temperature')

def temp_of(name):
    m = re.search('temperature_([0-9.]+)', name)
    return float(m.group(1)) if m else np.nan
TEMPS = {n: temp_of(n) for n in TMP}
print()
print('температуры:', sorted(TEMPS.values()))


свип temperature: плеч 20, кейсов 60

температуры: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]


In [2]:
rows = []
for n, d in TMP.items():
    js_ = pd.to_numeric(d['Judge_score'], errors='coerce')
    ps = participants(d)
    votes = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in ps})
    rows.append({'T': TEMPS[n], 'кейсов': len(d), 'с оценкой': int(js_.notna().sum()), 'участников': len(ps), 'полнота голосов R1': float(votes.notna().all(axis=1).mean()), 'Judge': float(js_.mean())})
Q = pd.DataFrame(rows).sort_values('T').set_index('T')
show(Q, 'Целостность прогонов по температуре')
GOOD = Q[(Q['кейсов'] >= 50) & (Q['полнота голосов R1'] > 0.8)].index.tolist()
print()
print('температур с полными данными: %d из %d' % (len(GOOD), len(Q)))
print('исключены:', [t for t in Q.index if t not in GOOD])



Целостность прогонов по температуре
------------------------------------------------------------
        кейсов  с оценкой  участников  полнота голосов R1  Judge
T                                                               
0.1000      60         60           3              0.9333 0.8128
0.2000      60         60           3              0.9333 0.8337
0.3000      60         60           3              0.9167 0.8118
0.4000      60         60           3              0.9667 0.8069
0.5000      60         60           3              0.8833 0.8118
0.6000      60         60           3              0.9333 0.8136
0.7000      60         60           3              0.9333 0.8087
0.8000      60         60           3              0.8833 0.8145
0.9000      60         60           3              0.9000 0.8208
1.0000      60         60           3              0.8833 0.8255
1.1000      60         60           3              0.8500 0.8186
1.2000      60         60           3              0.7167

In [3]:
def diversity_stats(d):
    t = true_label(d)
    ps = participants(d)
    lab = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in ps})
    corr_mat = pd.DataFrame({p: (lab[p] == t).astype(float).where(lab[p].notna() & t.notna()) for p in ps}).dropna()
    out = {'полных голосований': len(corr_mat)}
    if corr_mat.shape[1] >= 2 and len(corr_mat) >= 10:
        C = corr_mat.corr()
        rs = [C.iloc[i, j] for i, j in combinations(range(C.shape[0]), 2)]
        rbar = float(np.nanmean(rs))
        k = C.shape[1]
        out['ρ̄ ошибок'] = rbar
        out['k_eff'] = k / (1 + (k - 1) * rbar) if 1 + (k - 1) * rbar > 0 else np.nan
    else:
        out['ρ̄ ошибок'] = np.nan
        out['k_eff'] = np.nan
    full = lab.dropna()
    out['единогласие R1'] = float((full.nunique(axis=1) == 1).mean()) if len(full) else np.nan
    dis = []
    for a, b in combinations(ps, 2):
        s = lab[[a, b]].dropna()
        if len(s):
            dis.append(float((s[a] != s[b]).mean()))
    out['несогласованность'] = float(np.mean(dis)) if dis else np.nan
    rr = pd.to_numeric(d.get('delphi_rounds_executed'), errors='coerce')
    out['ср. раундов'] = float(rr.mean()) if rr is not None else np.nan
    return out
rows = []
for n, d in TMP.items():
    r = diversity_stats(d)
    r['T'] = TEMPS[n]
    r['Judge'] = float(pd.to_numeric(d['Judge_score'], errors='coerce').mean())
    rows.append(r)
D = pd.DataFrame(rows).sort_values('T').set_index('T')
show(D[['ρ̄ ошибок', 'k_eff', 'единогласие R1', 'несогласованность', 'ср. раундов', 'Judge']], 'Разнообразие и качество по температуре')
DG = D.loc[[t for t in D.index if t in GOOD]]
print()
print('ТРЕНДЫ по температурам с полными данными (n = %d точек):' % len(DG))
for col in ['ρ̄ ошибок', 'k_eff', 'единогласие R1', 'несогласованность', 'ср. раундов']:
    s = DG[col].dropna()
    if len(s) >= 4:
        r, p = stats.spearmanr(s.index, s.to_numpy())
        print('  %-22s rho с T = %+.3f   p = %s' % (col, r, fmt_p(p)))
print()
print('H17.1 подтверждается, если ρ̄ и единогласие падают с ростом T,')
print('а k_eff и несогласованность растут.')



Разнообразие и качество по температуре
------------------------------------------------------------
        ρ̄ ошибок  k_eff  единогласие R1  несогласованность  ср. раундов  Judge
T                                                                              
0.1000     0.5810 1.3875          0.7500             0.1627       1.8333 0.8128
0.2000     0.5190 1.4721          0.7321             0.1790       1.7000 0.8337
0.3000     0.4644 1.5553          0.6909             0.2045       1.7500 0.8118
0.4000     0.6532 1.3007          0.7931             0.1418       1.8167 0.8069
0.5000     0.5572 1.4188          0.7358             0.1743       1.7500 0.8118
0.6000     0.5799 1.3891          0.7500             0.1734       1.8333 0.8136
0.7000     0.5725 1.3986          0.7321             0.1742       1.7667 0.8087
0.8000     0.4387 1.5980          0.6981             0.1919       1.8500 0.8145
0.9000     0.6516 1.3026          0.8148             0.1357       1.7167 0.8208
1.0000     0.4726 1

In [4]:
FULL = D.dropna(subset=['несогласованность'], how='all')
for col in ['несогласованность', 'ср. раундов', 'Judge']:
    s = FULL[col].dropna()
    if len(s) >= 5:
        r, p = stats.spearmanr(s.index, s.to_numpy())
        print('  %-22s (полный диапазон, n=%2d)  rho с T = %+.3f   p = %s' % (col, len(s), r, fmt_p(p)))
print()
print('Полнота голосов R1 по зонам:')
zone = pd.cut(Q.index.to_series(), [0, 1.25, 2.1], labels=['0.1-1.2', '1.3-2.0'])
show(Q.groupby(zone, observed=True).agg(точек=('Judge', 'size'), кейсов=('кейсов', 'mean'), полнота=('полнота голосов R1', 'mean'), Judge=('Judge', 'mean')), None)
print()
print('ВЫВОД ПО ШАГУ 2. В рабочем диапазоне (0.1-1.2), где данные полны,')
print('температура НЕ управляет корреляцией ошибок: ρ̄ колеблется в коридоре')
print('0.44-0.65 без тренда. Выше 1.2 растёт видимая несогласованность, но')
print('одновременно рушится разбор ответов и падает число раундов — то есть')
print('это деградация, а не разнообразие.')


  несогласованность      (полный диапазон, n=18)  rho с T = +0.699   p = 0.001
  ср. раундов            (полный диапазон, n=18)  rho с T = -0.698   p = 0.001
  Judge                  (полный диапазон, n=18)  rho с T = -0.521   p = 0.027

Полнота голосов R1 по зонам:
         точек  кейсов  полнота  Judge
T                                     
0.1-1.2     12 60.0000   0.8944 0.8171
1.3-2.0      8 45.6250   0.0167 0.6984

ВЫВОД ПО ШАГУ 2. В рабочем диапазоне (0.1-1.2), где данные полны,
температура НЕ управляет корреляцией ошибок: ρ̄ колеблется в коридоре
0.44-0.65 без тренда. Выше 1.2 растёт видимая несогласованность, но
одновременно рушится разбор ответов и падает число раундов — то есть
это деградация, а не разнообразие.


In [5]:
parts = []
for n, d in TMP.items():
    if TEMPS[n] not in GOOD:
        continue
    s = pd.to_numeric(d['Judge_score'], errors='coerce').dropna()
    parts.append(pd.DataFrame({'y': s.to_numpy(), 'T': TEMPS[n], 'case': s.index}))
L = pd.concat(parts, ignore_index=True)
L['T_c'] = L['T'] - L['T'].mean()
L['T2'] = L['T_c'] ** 2
CD = pd.get_dummies(L['case'], prefix='c', drop_first=True).astype(float)
print('наблюдений: %d  (температур %d × кейсов ~%d)' % (len(L), L['T'].nunique(), L['case'].nunique()))
g = L['case'].astype('category').cat.codes
m_lin = sm.OLS(L['y'], sm.add_constant(pd.concat([L[['T_c']], CD], axis=1))).fit(cov_type='cluster', cov_kwds={'groups': g})
m_qua = sm.OLS(L['y'], sm.add_constant(pd.concat([L[['T_c', 'T2']], CD], axis=1))).fit(cov_type='cluster', cov_kwds={'groups': g})
res = pd.DataFrame({'линейная коэф': m_lin.params, 'p': m_lin.pvalues, 'квадратичная коэф': m_qua.params, 'p ': m_qua.pvalues})
show(res.loc[[i for i in ['const', 'T_c', 'T2'] if i in res.index]], 'Judge ~ температура (с фиксированным эффектом кейса)')
b2 = m_qua.params.get('T2', 0)
if b2 < 0:
    opt = -m_qua.params['T_c'] / (2 * b2) + L['T'].mean()
    print('\nоптимум параболы: T = %.2f' % opt)
    print('диапазон данных : T = %.1f ... %.1f' % (L['T'].min(), L['T'].max()))
    print('оптимум внутри диапазона: %s' % ('да' if L['T'].min() < opt < L['T'].max() else 'НЕТ (экстраполяция)'))
else:
    print('\nквадратичный член неотрицателен — внутреннего максимума нет')


наблюдений: 660  (температур 11 × кейсов ~60)

Judge ~ температура (с фиксированным эффектом кейса)
------------------------------------------------------------
       линейная коэф      p  квадратичная коэф     p 
const         0.8636 0.0000             0.8596 0.0000
T_c           0.0032 0.7525             0.0032 0.7527
T2               NaN    NaN             0.0402 0.3218

квадратичный член неотрицателен — внутреннего максимума нет


In [6]:
D2 = DG.copy()
D2['группа T'] = pd.cut(D2.index.to_series(), [0, 0.5, 1.0, 1.5, 2.1], labels=['низкая 0.1-0.5', 'средняя 0.6-1.0', 'высокая 1.1-1.5', 'очень высокая 1.6-2.0'])
show(D2.groupby('группа T', observed=True).agg(точек=('Judge', 'size'), Judge=('Judge', 'mean'), k_eff=('k_eff', 'mean'), единогласие=('единогласие R1', 'mean')), 'Качество и разнообразие по группам температур')



Качество и разнообразие по группам температур
------------------------------------------------------------
                 точек  Judge  k_eff  единогласие
группа T                                         
низкая 0.1-0.5       5 0.8154 1.4269       0.7404
средняя 0.6-1.0      5 0.8166 1.4461       0.7386
высокая 1.1-1.5      1 0.8186 1.3895       0.7647


In [7]:
rows = []
for pred in ['T', 'ρ̄ ошибок', 'k_eff', 'единогласие R1', 'несогласованность']:
    s = DG.reset_index()[[pred if pred != 'T' else 'T', 'Judge']].dropna()
    if len(s) < 5:
        continue
    r, p = stats.spearmanr(s.iloc[:, 0], s['Judge'])
    rp, pp = stats.pearsonr(s.iloc[:, 0], s['Judge'])
    rows.append({'предиктор': pred, 'n точек': len(s), 'Спирмен': r, 'p (Спирмен)': p, 'Пирсон': rp, 'p (Пирсон)': pp})
show(pd.DataFrame(rows).set_index('предиктор'), 'Что предсказывает качество панели')
print()
print('ПРЕДУПРЕЖДЕНИЕ: при n = %d точках значимой на уровне 0.05 будет' % len(DG))
print('ранговая корреляция примерно от 0.45 по модулю. Смотреть надо не')
print('только на p, но и на величину и согласованность знаков.')



Что предсказывает качество панели
------------------------------------------------------------
                   n точек  Спирмен  p (Спирмен)  Пирсон  p (Пирсон)
предиктор                                                           
T                       11   0.3545       0.2847  0.1350      0.6924
ρ̄ ошибок               11  -0.2455       0.4669 -0.2661      0.4290
k_eff                   11   0.2455       0.4669  0.2552      0.4487
единогласие R1          11  -0.0412       0.9043 -0.1011      0.7675
несогласованность       11   0.0909       0.7904  0.1155      0.7353

ПРЕДУПРЕЖДЕНИЕ: при n = 11 точках значимой на уровне 0.05 будет
ранговая корреляция примерно от 0.45 по модулю. Смотреть надо не
только на p, но и на величину и согласованность знаков.


In [8]:
rows = []
for n, d in TMP.items():
    if TEMPS[n] not in GOOD:
        continue
    rr = pd.to_numeric(d.get('delphi_rounds_executed'), errors='coerce')
    esr = d.get('delphi_early_stop_reason')
    r = {'T': TEMPS[n], 'ср. раундов': float(rr.mean()), 'доля стоп после R1': float((rr == 1).mean()), 'доля 3 раунда': float((rr == 3).mean())}
    rows.append(r)
R = pd.DataFrame(rows).sort_values('T').set_index('T')
show(R, 'Поведение протокола по температуре')
s = R['ср. раундов'].dropna()
r, p = stats.spearmanr(s.index, s.to_numpy())
print('\nсвязь «температура → среднее число раундов»: rho = %+.3f, p = %s' % (r, fmt_p(p)))
print('Положительная связь означает: разнообразие действительно заставляет')
print('протокол работать дольше — то есть рычаг влияет не только на ответы,')
print('но и на динамику процедуры.')



Поведение протокола по температуре
------------------------------------------------------------
        ср. раундов  доля стоп после R1  доля 3 раунда
T                                                     
0.1000       1.8333              0.4500         0.2833
0.2000       1.7000              0.4667         0.1667
0.3000       1.7500              0.3833         0.1333
0.4000       1.8167              0.4500         0.2667
0.5000       1.7500              0.5000         0.2500
0.6000       1.8333              0.4500         0.2833
0.7000       1.7667              0.4500         0.2167
0.8000       1.8500              0.4167         0.2667
0.9000       1.7167              0.5000         0.2167
1.0000       1.6833              0.5167         0.2000
1.1000       1.7167              0.4500         0.1667

связь «температура → среднее число раундов»: rho = -0.297, p = 0.374
Положительная связь означает: разнообразие действительно заставляет
протокол работать дольше — то есть рычаг влияет не

In [9]:
span_T = DG['ρ̄ ошибок'].max() - DG['ρ̄ ошибок'].min()
print('размах ρ̄ при варьировании ТЕМПЕРАТУРЫ (состав фиксирован): %.3f' % span_T)
print('   от %.3f (T = %.1f) до %.3f (T = %.1f)' % (DG['ρ̄ ошибок'].min(), DG['ρ̄ ошибок'].idxmin(), DG['ρ̄ ошибок'].max(), DG['ρ̄ ошибок'].idxmax()))
print()
print('размах ρ̄ при варьировании СОСТАВА (из C15, температура фиксирована):')
print('   от 0.582 (P-base) до 0.616 (P-weak)  ->  0.034')
print()
print('Если размах по температуре заметно больше, это означает, что')
print('температура — более сильный рычаг разнообразия, чем выбор моделей.')
print('Практическое следствие: дешевле крутить температуру, чем искать')
print('«разные» модели.')


размах ρ̄ при варьировании ТЕМПЕРАТУРЫ (состав фиксирован): 0.215


   от 0.439 (T = 0.8) до 0.653 (T = 0.4)

размах ρ̄ при варьировании СОСТАВА (из C15, температура фиксирована):
   от 0.582 (P-base) до 0.616 (P-weak)  ->  0.034

Если размах по температуре заметно больше, это означает, что
температура — более сильный рычаг разнообразия, чем выбор моделей.
Практическое следствие: дешевле крутить температуру, чем искать
«разные» модели.


In [10]:
save(Q.reset_index(), 'temperature_and_response_diversity_integrity.csv')
save(D.reset_index(), 'temperature_and_response_diversity_diversity_by_temp.csv')
save(R.reset_index(), 'temperature_and_response_diversity_protocol_behaviour.csv')


сохранено: out/temperature_and_response_diversity_integrity.csv  (20 строк)
сохранено: out/temperature_and_response_diversity_diversity_by_temp.csv  (20 строк)
сохранено: out/temperature_and_response_diversity_protocol_behaviour.csv  (11 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\protocol_design\\results\\temperature_and_response_diversity_protocol_behaviour.csv'